# 第4章: 言語解析

問題30から問題35までは、以下の文章`text`（太宰治の『走れメロス』の冒頭部分）に対して、言語解析を実施せよ。問題36から問題39までは、国家を説明した文書群（日本語版ウィキペディア記事から抽出したテキスト群）をコーパスとして、言語解析を実施せよ。

In [12]:
text = """
メロスは激怒した。
必ず、かの邪智暴虐の王を除かなければならぬと決意した。
メロスには政治がわからぬ。
メロスは、村の牧人である。
笛を吹き、羊と遊んで暮して来た。
けれども邪悪に対しては、人一倍に敏感であった。
"""

## 30. 動詞
文章`text`に含まれる動詞をすべて表示せよ。

In [13]:
!apt-get -q -y install mecab libmecab-dev mecab-ipadic-utf8
!pip install mecab-python3
!pip install unidic-lite

Reading package lists...
Building dependency tree...
Reading state information...
libmecab-dev is already the newest version (0.996-14build9).
mecab-ipadic-utf8 is already the newest version (2.7.0-20070801+main-3).
mecab is already the newest version (0.996-14build9).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.


In [18]:
import MeCab

tagger = MeCab.Tagger()
me_text = tagger.parse(text)
print(me_text)

メロス	メロス	メロス	メロス-melos	名詞-普通名詞-一般			1
は	ワ	ハ	は	助詞-係助詞			
激怒	ゲキド	ゲキド	激怒	名詞-普通名詞-サ変可能			1
し	シ	スル	為る	動詞-非自立可能	サ行変格	連用形-一般	0
た	タ	タ	た	助動詞	助動詞-タ	終止形-一般	
。			。	補助記号-句点			
必ず	カナラズ	カナラズ	必ず	副詞			0
、			、	補助記号-読点			
かの	カノ	カノ	彼の	連体詞			1
邪智	ジャチ	ジャチ	邪知	名詞-普通名詞-一般			1
暴虐	ボーギャク	ボウギャク	暴虐	名詞-普通名詞-形状詞可能			0
の	ノ	ノ	の	助詞-格助詞			
王	オー	オウ	王	名詞-普通名詞-一般			1
を	オ	ヲ	を	助詞-格助詞			
除か	ノゾカ	ノゾク	除く	動詞-一般	五段-カ行	未然形-一般	0
なけれ	ナケレ	ナイ	ない	助動詞	助動詞-ナイ	仮定形-一般	
ば	バ	バ	ば	助詞-接続助詞			
なら	ナラ	ナル	成る	動詞-非自立可能	五段-ラ行	未然形-一般	1
ぬ	ヌ	ズ	ず	助動詞	助動詞-ヌ	終止形-一般	
と	ト	ト	と	助詞-格助詞			
決意	ケツイ	ケツイ	決意	名詞-普通名詞-サ変可能			1,2
し	シ	スル	為る	動詞-非自立可能	サ行変格	連用形-一般	0
た	タ	タ	た	助動詞	助動詞-タ	終止形-一般	
。			。	補助記号-句点			
メロス	メロス	メロス	メロス-melos	名詞-普通名詞-一般			1
に	ニ	ニ	に	助詞-格助詞			
は	ワ	ハ	は	助詞-係助詞			
政治	セージ	セイジ	政治	名詞-普通名詞-一般			0
が	ガ	ガ	が	助詞-格助詞			
わから	ワカラ	ワカル	分かる	動詞-一般	五段-ラ行	未然形-一般	2
ぬ	ヌ	ズ	ず	助動詞	助動詞-ヌ	終止形-一般	
。			。	補助記号-句点			
メロス	メロス	メロス	メロス-melos	名詞-普通名詞-一般			1
は	ワ	ハ	は	助詞-係助詞			
、			、	補助記号-読点			
村	ムラ	ムラ	村	名詞-普通名詞-一般			2
の	ノ	ノ	の	助詞-格助詞			
牧人	ボクジン	ボクジン	牧人	名詞-普通名詞-一般			0
で	デ	ダ	だ	助動詞	助動詞-ダ	連用形-一般	
ある	ア

In [17]:
node = tagger.parseToNode(text)
while node:
  word = node.surface
  feature = node.feature.split(',')
  if feature[0] == '動詞':
    print(word)
  node = node.next

し
除か
なら
し
わから
ある
吹き
遊ん
暮し
来
対し
あっ


## 31. 動詞の原型
文章`text`に含まれる動詞と、その原型をすべて表示せよ。

In [20]:
node = tagger.parseToNode(text)
while node:
  word = node.surface
  feature = node.feature.split(',')
  if feature[0] == '動詞':
    print(word, feature[7])
  node = node.next

し 為る
除か 除く
なら 成る
し 為る
わから 分かる
ある 有る
吹き 吹く
遊ん 遊ぶ
暮し 暮らす
来 来る
対し 対する
あっ 有る


## 32. 「AのB」
文章`text`において、2つの名詞が「の」で連結されている名詞句をすべて抽出せよ。

In [27]:
node = tagger.parseToNode(text)
while node:

  feature = node.feature.split(',')

  if node.prev is not None:
    pre_feature = node.prev.feature.split(',')
  #連続する名詞はつなげる
  if feature[0] == '名詞' and pre_feature[0] == '名詞':
    word += node.surface
  else:
    word = node.surface

  if node.next is not None and node.next.next is not None:
    next_word = node.next.surface
    next_feature = node.next.feature.split(',')

    if next_feature[0] == '助詞' and next_word == "の":
      print(word, next_word, node.next.next.surface)
  node = node.next

邪智暴虐 の 王
村 の 牧人


## 33. 係り受け解析

文章`text`に係り受け解析を適用し、係り元と係り先のトークン（形態素や文節などの単位）をタブ区切り形式ですべて抽出せよ。

## 34. 主述の関係
文章`text`において、「メロス」が主語であるときの述語を抽出せよ。

## 35. 係り受け木
「メロスは激怒した。」の係り受け木を可視化せよ。

## 36. 単語の出現頻度

問題36から39までは、Wikipediaの記事を以下のフォーマットで書き出したファイル[jawiki-country.json.gz](/data/jawiki-country.json.gz)をコーパスと見なし、統計的な分析を行う。

* 1行に1記事の情報がJSON形式で格納される
* 各行には記事名が"title"キーに、記事本文が"text"キーの辞書オブジェクトに格納され、そのオブジェクトがJSON形式で書き出される
* ファイル全体はgzipで圧縮される

まず、第3章の処理内容を参考に、Wikipedia記事からマークアップを除去し、各記事のテキストを抽出せよ。そして、コーパスにおける単語（形態素）の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 37. 名詞の出現頻度
コーパスにおける名詞の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 38. TF・IDF
日本に関する記事における名詞のTF・IDFスコアを求め、TF・IDFスコア上位20語とそのTF, IDF, TF・IDFを表示せよ。

## 39. Zipfの法則
コーパスにおける単語の出現頻度順位を横軸、その出現頻度を縦軸として、両対数グラフをプロットせよ。